# 1D spectral element validation: non-constant load and non-homogeneous Dirichlet data

We solve

$$
-u''(x)=1+x^2,\qquad x\in[0,1],
$$

with

$$
u(0)=1,\qquad u(1)=2.
$$

The exact solution is

$$
u(x)
=
1+\frac{19}{12}x-\frac{x^2}{2}-\frac{x^4}{12}.
$$

With polynomial order $p=4$, the exact solution belongs to the approximation
space. Therefore this is a direct test of the transformation-based spectral
element layer rather than of discretization error.

In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Printf

p = 4          # polynomial order
L = 1.0        # element length
n = 3          # start with exactly one element

line_mesh(lx=L, n=n, order=p)

## 1. Read the high-order Gmsh element

A crucial point is that we do **not** assume a local ordering such as
`linspace(-1,1,p+1)`.

For example, Gmsh typically stores the two end nodes first and the internal
high-order nodes afterwards. Therefore the spectral transformation must use
the reference coordinates returned by `gmsh.model.mesh.getElementProperties`.

In [2]:
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(1, -1)

@assert length(elemTypes) == 1 "This prototype expects a single 1D element type."

etype = elemTypes[1]

elementName, elementDim, p_gmsh, numNodes, localNodeCoord, numPrimaryNodes =
    gmsh.model.mesh.getElementProperties(etype)

@assert p_gmsh == p
@assert length(localNodeCoord) == numNodes
@assert length(elemTags[1]) == n

ξ_gmsh = Float64.(localNodeCoord)

connectivity =
    permutedims(reshape(Int.(elemNodeTags[1]), numNodes, :))

nodeTags, nodeCoord, _ = gmsh.model.mesh.getNodes()
nodeTags = Int.(nodeTags)
XYZ = permutedims(reshape(Float64.(nodeCoord), 3, :))

tagToRow = Dict(tag => i for (i, tag) in enumerate(nodeTags))

# Physical coordinates of the first element in Gmsh local-node ordering.
X_gmsh = XYZ[[tagToRow[tag] for tag in connectivity[1, :]], :]

println("Element type : ", elementName)
println("Order        : ", p_gmsh)
println("Number nodes : ", numNodes)
println()
println("Gmsh local ordering:")
display(hcat(connectivity[1, :], ξ_gmsh, X_gmsh[:, 1]))

Element type : Line 5
Order        : 4
Number nodes : 5

Gmsh local ordering:


5×3 Matrix{Float64}:
 1.0  -1.0  0.0
 3.0   1.0  0.333333
 5.0  -0.5  0.0833333
 6.0   0.0  0.166667
 7.0   0.5  0.25

## 2. Legendre–Gauss–Lobatto nodes and weights

For polynomial order $p$, the GLL rule contains $p+1$ points:

- $\xi_0=-1$,
- $\xi_p=+1$,
- the $p-1$ interior roots of $P_p'(\xi)$.

The weights are

$$
w_i =
\frac{2}
{p(p+1)\,[P_p(\xi_i)]^2}.
$$

The implementation is general in $p$. For the first LowLevelFEM implementation
we can still decide to document/test only a conservative range such as
$1\le p\le10$.

In [3]:
"""
    legendre_pair(p, x)

Return `(Pp, Ppm1)`, where `Pp = P_p(x)` and `Ppm1 = P_{p-1}(x)`.
"""
function legendre_pair(p::Int, x::Real)
    p ≥ 0 || throw(ArgumentError("p must be non-negative."))

    p == 0 && return (1.0, 0.0)

    P0 = 1.0
    P1 = Float64(x)

    p == 1 && return (P1, P0)

    for k in 2:p
        Pk = ((2k - 1) * x * P1 - (k - 1) * P0) / k
        P0, P1 = P1, Pk
    end

    return P1, P0
end


"""
    gll(p; tol=1e-14, maxiter=100)

Compute the `p+1` Legendre–Gauss–Lobatto nodes and quadrature weights
on `[-1,1]`.
"""
function gll(p::Int; tol=1e-14, maxiter=100)
    p ≥ 1 || throw(ArgumentError("p must be at least one."))

    ξ = zeros(Float64, p + 1)
    ξ[1] = -1.0
    ξ[end] = 1.0

    # Chebyshev–Lobatto points are excellent Newton initial guesses.
    for k in 1:p-1
        x = -cos(pi * k / p)

        for _ in 1:maxiter
            Pp, Ppm1 = legendre_pair(p, x)

            dPp =
                p * (Ppm1 - x * Pp) / (1 - x^2)

            # Legendre differential equation:
            # (1-x^2) P'' - 2x P' + p(p+1)P = 0
            ddPp =
                (2x * dPp - p * (p + 1) * Pp) / (1 - x^2)

            Δx = dPp / ddPp
            x -= Δx

            abs(Δx) ≤ tol * max(1.0, abs(x)) && break
        end

        ξ[k + 1] = x
    end

    sort!(ξ)

    w = similar(ξ)
    for i in eachindex(ξ)
        Pp, _ = legendre_pair(p, ξ[i])
        w[i] = 2 / (p * (p + 1) * Pp^2)
    end

    return ξ, w
end


ξ_gll, w_gll = gll(p)

println("GLL nodes:")
display(ξ_gll)

println("GLL weights:")
display(w_gll)

@show sum(w_gll)

GLL nodes:


5-element Vector{Float64}:
 -1.0
 -0.6546536707079772
  0.0
  0.6546536707079772
  1.0

GLL weights:


5-element Vector{Float64}:
 0.1
 0.5444444444444444
 0.7111111111111111
 0.5444444444444444
 0.1

sum(w_gll) = 2.0


2.0

## 3. Lagrange interpolation and transformation matrices

Let $\mathbf u_G$ denote nodal values in the GLL basis and
$\mathbf u_E$ nodal values in the original Gmsh basis.

The matrix

$$
T_{ij}=L_j^\mathrm{GLL}(\xi_i^\mathrm{gmsh})
$$

gives

$$
\mathbf u_E=T\mathbf u_G.
$$

The reverse interpolation matrix is

$$
R_{ij}=L_j^\mathrm{gmsh}(\xi_i^\mathrm{GLL}),
$$

so for two bases spanning the same $P_p$ polynomial space,

$$
RT \approx I,\qquad TR\approx I.
$$

In [4]:
"""
    barycentric_weights(nodes)

Compute barycentric interpolation weights for distinct interpolation nodes.
"""
function barycentric_weights(nodes::AbstractVector)
    n = length(nodes)
    w = ones(Float64, n)

    for j in 1:n
        for k in 1:n
            j == k && continue
            w[j] /= nodes[j] - nodes[k]
        end
    end

    return w
end


"""
    lagrange_matrices(nodes, points; atol=1e-13)

Return `(N, D)`, where `N[q,j]` is the `j`th Lagrange polynomial evaluated
at `points[q]`, and `D[q,j]` is its derivative with respect to the reference
coordinate.
"""
function lagrange_matrices(
    nodes::AbstractVector,
    points::AbstractVector;
    atol=1e-13
)
    nodes = Float64.(nodes)
    points = Float64.(points)

    n = length(nodes)
    m = length(points)

    bw = barycentric_weights(nodes)

    N = zeros(Float64, m, n)
    D = zeros(Float64, m, n)

    for q in 1:m
        x = points[q]

        hit = findfirst(j -> isapprox(x, nodes[j]; atol=atol, rtol=0.0), 1:n)

        if hit !== nothing
            i = hit
            N[q, i] = 1.0

            for j in 1:n
                j == i && continue
                D[q, j] =
                    bw[j] / (bw[i] * (nodes[i] - nodes[j]))
            end

            D[q, i] = -sum(D[q, j] for j in 1:n if j != i)
        else
            dx = x .- nodes
            a = bw ./ dx
            S = sum(a)

            Nq = a ./ S
            N[q, :] .= Nq

            S2 = sum(bw ./ dx.^2)
            D[q, :] .= Nq .* (S2 / S .- 1.0 ./ dx)
        end
    end

    return N, D
end


T, _ = lagrange_matrices(ξ_gll, ξ_gmsh)
R, _ = lagrange_matrices(ξ_gmsh, ξ_gll)

Iref = Matrix{Float64}(I, numNodes, numNodes)

println("T : GLL DOFs -> Gmsh DOFs")
display(round.(T, digits=10))

println("R : Gmsh DOFs -> GLL DOFs")
display(round.(R, digits=10))

@show norm(R * T - Iref, Inf)
@show norm(T * R - Iref, Inf)

T : GLL DOFs -> Gmsh DOFs


5×5 Matrix{Float64}:
  1.0         0.0       0.0      0.0        0.0
  0.0         0.0       0.0      0.0        1.0
 -0.117188    0.884032  0.3125  -0.118407   0.0390625
  0.0         0.0       1.0      0.0        0.0
  0.0390625  -0.118407  0.3125   0.884032  -0.117188

R : Gmsh DOFs -> GLL DOFs


5×5 Matrix{Float64}:
  1.0         0.0        0.0        0.0       0.0
  0.128955   -0.0269146  1.15184   -0.408163  0.154277
  0.0         0.0        0.0        1.0       0.0
 -0.0269146   0.128955   0.154277  -0.408163  1.15184
  0.0         1.0        0.0        0.0       0.0

norm(R * T - Iref, Inf) = 3.3306690738754696e-16
norm(T * R - Iref, Inf) = 3.3306690738754696e-16


3.3306690738754696e-16

In [5]:
"""
    insert_unique!(A, i, j, value; atol=1e-12)

Insert an entry into a global transformation matrix represented by a
dictionary. If the entry already exists, verify that the duplicated
contribution is identical.

This is needed at shared element endpoints: adjacent elements define the
same transformation row there, which must not be added twice.
"""
function insert_unique!(
    A::Dict{Tuple{Int,Int},Float64},
    i::Int,
    j::Int,
    value::Real;
    atol=1e-12
)
    abs(value) < atol && return

    key = (i, j)
    value = Float64(value)

    if haskey(A, key)
        isapprox(A[key], value; atol=atol, rtol=atol) ||
            error(
                "Inconsistent transformation at global entry " *
                "($i,$j): $(A[key]) vs. $value"
            )
    else
        A[key] = value
    end

    return
end


"""
    sparse_from_dict(A, m, n)

Convert a dictionary of matrix entries into a sparse matrix.
"""
function sparse_from_dict(
    A::Dict{Tuple{Int,Int},Float64},
    m::Int,
    n::Int
)
    I = Int[]
    J = Int[]
    V = Float64[]

    sizehint!(I, length(A))
    sizehint!(J, length(A))
    sizehint!(V, length(A))

    for ((i, j), value) in A
        push!(I, i)
        push!(J, j)
        push!(V, value)
    end

    return sparse(I, J, V, m, n)
end


# ------------------------------------------------------------------
# Global spectral transformation
# ------------------------------------------------------------------

ndof = length(nodeTags)

# Gmsh node tag -> global matrix row/column
tagToDof = Dict(tag => i for (i, tag) in enumerate(nodeTags))

# GLL local ordering is monotonically increasing in ξ.
perm_gll = sortperm(ξ_gmsh)

Tentries = Dict{Tuple{Int,Int},Float64}()
Rentries = Dict{Tuple{Int,Int},Float64}()

for e in axes(connectivity, 1)

    # Gmsh local ordering
    gmsh_tags = vec(connectivity[e, :])

    # Same abstract global node tags, but ordered according to GLL ξ.
    gll_tags = gmsh_tags[perm_gll]

    gmsh_dofs = [tagToDof[tag] for tag in gmsh_tags]
    gll_dofs  = [tagToDof[tag] for tag in gll_tags]

    for i in 1:numNodes
        for j in 1:numNodes
            insert_unique!(
                Tentries,
                gmsh_dofs[i],
                gll_dofs[j],
                T[i, j]
            )

            insert_unique!(
                Rentries,
                gll_dofs[i],
                gmsh_dofs[j],
                R[i, j]
            )
        end
    end
end

T_global = sparse_from_dict(Tentries, ndof, ndof)
R_global = sparse_from_dict(Rentries, ndof, ndof)

I_global = spdiagm(0 => ones(ndof))

println("Global transformation:")
@show size(T_global)
@show nnz(T_global)
@show size(R_global)
@show nnz(R_global)

println()
println("Inverse checks:")
@show norm(Matrix(R_global * T_global - I_global), Inf)
@show norm(Matrix(T_global * R_global - I_global), Inf)

Global transformation:
size(T_global) = (13, 13)
nnz(T_global) = 37
size(R_global) = (13, 13)
nnz(R_global) = 37

Inverse checks:
norm(Matrix(R_global * T_global - I_global), Inf) = 4.440892098500626e-16
norm(Matrix(T_global * R_global - I_global), Inf) = 4.440892098500626e-16


4.440892098500626e-16

## 4. Physical coordinates of the GLL nodes

The Gmsh high-order geometry is retained. We only evaluate that geometry at
the GLL reference coordinates:

$$
\mathbf X_\mathrm{GLL}=R\,\mathbf X_\mathrm{gmsh}.
$$

Thus Gmsh remains responsible for the mesh topology, physical groups and
element connectivity; the spectral layer only introduces a different nodal
representation of the same polynomial geometry.

In [6]:
# ------------------------------------------------------------------
# Global physical coordinates of the GLL DOFs
# ------------------------------------------------------------------

X_gll_global = fill(NaN, ndof, 3)
assigned = falses(ndof)

for e in axes(connectivity, 1)

    gmsh_tags = vec(connectivity[e, :])
    gll_tags  = gmsh_tags[perm_gll]

    gmsh_dofs = [tagToDof[tag] for tag in gmsh_tags]
    gll_dofs  = [tagToDof[tag] for tag in gll_tags]

    X_e = XYZ[gmsh_dofs, :]

    # Same geometry, evaluated at GLL nodes.
    X_gll_e = R * X_e

    for i in 1:numNodes
        gi = gll_dofs[i]

        if assigned[gi]
            norm(X_gll_global[gi, :] - X_gll_e[i, :]) < 1e-12 ||
                error("Inconsistent GLL coordinate at shared node.")
        else
            X_gll_global[gi, :] .= X_gll_e[i, :]
            assigned[gi] = true
        end
    end
end

@assert all(assigned)

# Display nodes ordered by their spectral physical coordinate.
ix = sortperm(X_gll_global[:, 1])

println("Global Gmsh and GLL coordinates:")
display(
    hcat(
        nodeTags[ix],
        XYZ[ix, 1],
        X_gll_global[ix, 1]
    )
)

Global Gmsh and GLL coordinates:


13×3 Matrix{Float64}:
  1.0  0.0        0.0
  5.0  0.0833333  0.0575577
  6.0  0.166667   0.166667
  7.0  0.25       0.275776
  3.0  0.333333   0.333333
  8.0  0.416667   0.390891
  9.0  0.5        0.5
 10.0  0.583333   0.609109
  4.0  0.666667   0.666667
 11.0  0.75       0.724224
 12.0  0.833333   0.833333
 13.0  0.916667   0.942442
  2.0  1.0        1.0

## Put the global transformation into LowLevelFEM DOF ordering

For this scalar problem, LowLevelFEM uses the contiguous Gmsh node tags as DOF
indices. The global transformation is reordered accordingly.

In [7]:
@assert sort(nodeTags) == collect(1:ndof) "This test assumes contiguous Gmsh node tags 1:ndof."

ref_order = [tagToDof[tag] for tag in 1:ndof]

T_dof = T_global[ref_order, ref_order]
R_dof = R_global[ref_order, ref_order]

x_dof = [XYZ[tagToRow[tag], 1] for tag in 1:ndof]
x_gll_dof = X_gll_global[ref_order, 1]

@show norm(Matrix(R_dof * T_dof - I_global), Inf)
@show norm(Matrix(T_dof * R_dof - I_global), Inf)

norm(Matrix(R_dof * T_dof - I_global), Inf) = 4.440892098500626e-16
norm(Matrix(T_dof * R_dof - I_global), Inf) = 4.440892098500626e-16


4.440892098500626e-16

## Ordinary LowLevelFEM solution

The source term

$$
q(x)=1+x^2
$$

is represented as a LowLevelFEM `ScalarField`. The stiffness matrix, load
vector, and boundary conditions are assembled with the standard API.

In [8]:
mat = Material("body")

P = Problem(
    [mat];
    type=:ScalarField,
    dim=1,
    field=:u
)

source(x, y, z) = 1.0 + x^2

q = ScalarField(P, "body", source)

K = ∫(
    Grad(P) ⋅ Grad(P);
    Ω="body"
)

f = ∫(
    P ⋅ q;
    Ω="body"
)

support = [
    BoundaryCondition(
        "left";
        problem=P,
        u=1.0
    ),
    BoundaryCondition(
        "right";
        problem=P,
        u=2.0
    )
]

u_fem = solveField(
    Symmetric(K),
    f;
    support=support
)

nodal ScalarField
[1.0; 2.0; … ; 1.9320344650205794; 1.9724111046810715;;]

## Spectral solution

The existing LLFEM matrices and vector are transformed as

$$
K_G=T^\mathsf{T}KT,
\qquad
f_G=T^\mathsf{T}f.
$$

The same `solveField` routine and the same non-homogeneous boundary conditions
are used in the GLL basis.

In [9]:
K_sem = SystemMatrix(
    sparse(T_dof' * K.A * T_dof),
    P
)

f_sem_a = T_dof' * f.a

f_sem = ScalarField(
    [],
    Matrix(f_sem_a),
    f.t,
    [],
    f.nsteps,
    f.type,
    P
)

u_gll = solveField(
    Symmetric(K_sem),
    f_sem;
    support=support
)

u_back_a = T_dof * u_gll.a

u_sem = ScalarField(
    [],
    Matrix(u_back_a),
    u_gll.t,
    [],
    u_gll.nsteps,
    u_gll.type,
    P
)

nodal ScalarField
[1.0; 2.0; … ; 1.9320344650205747; 1.9724111046810697;;]

## Compare FEM, SEM, and the exact solution

The checks below compare

- the ordinary FEM solution with the back-transformed SEM solution,
- both nodal solutions with the exact solution at Gmsh nodes,
- the spectral coefficients directly with the exact solution at physical GLL
  nodes.

In [10]:
u_exact(x) =
    1.0 +
    (19.0 / 12.0) * x -
    x^2 / 2.0 -
    x^4 / 12.0

u_exact_gmsh = u_exact.(x_dof)
u_exact_gll = u_exact.(x_gll_dof)

println("Solution errors:")
@show norm(u_fem.a - u_sem.a, Inf)
@show norm(vec(u_fem.a[:, 1]) - u_exact_gmsh, Inf)
@show norm(vec(u_sem.a[:, 1]) - u_exact_gmsh, Inf)
@show norm(vec(u_gll.a[:, 1]) - u_exact_gll, Inf)

println()
println("Boundary values:")

left_i = argmin(x_dof)
right_i = argmax(x_dof)

@show u_fem.a[left_i, 1]
@show u_fem.a[right_i, 1]
@show u_sem.a[left_i, 1]
@show u_sem.a[right_i, 1]

Solution errors:
norm(u_fem.a - u_sem.a, Inf) = 9.325873406851315e-15
norm(vec(u_fem.a[:, 1]) - u_exact_gmsh, Inf) = 5.10702591327572e-15
norm(vec(u_sem.a[:, 1]) - u_exact_gmsh, Inf) = 1.3766765505351941e-14
norm(vec(u_gll.a[:, 1]) - u_exact_gll, Inf) = 1.3766765505351941e-14

Boundary values:
u_fem.a[left_i, 1] = 1.0
u_fem.a[right_i, 1] = 2.0
u_sem.a[left_i, 1] = 1.0
u_sem.a[right_i, 1] = 2.0


2.0

## Nodal values

Columns:

1. $x$,
2. ordinary LowLevelFEM solution,
3. SEM solution after back transformation,
4. exact solution.

In [11]:
ix = sortperm(x_dof)

display(
    hcat(
        x_dof[ix],
        vec(u_fem.a[ix, 1]),
        vec(u_sem.a[ix, 1]),
        u_exact_gmsh[ix]
    )
)

13×4 Matrix{Float64}:
 0.0        1.0      1.0      1.0
 0.0833333  1.12847  1.12847  1.12847
 0.166667   1.24994  1.24994  1.24994
 0.25       1.36426  1.36426  1.36426
 0.333333   1.47119  1.47119  1.47119
 0.416667   1.5704   1.5704   1.5704
 0.5        1.66146  1.66146  1.66146
 0.583333   1.74382  1.74382  1.74382
 0.666667   1.81687  1.81687  1.81687
 0.75       1.87988  1.87988  1.87988
 0.833333   1.93203  1.93203  1.93203
 0.916667   1.97241  1.97241  1.97241
 1.0        2.0      2.0      2.0

## Interpretation

If all errors are at round-off level, the transformation-based SEM layer
handles a non-constant distributed source and non-homogeneous Dirichlet data
without modifying either the LowLevelFEM weak-form assembly or `solveField`.

The complete static workflow is

$$
K,f
\longrightarrow
T^\mathsf{T}KT,\;T^\mathsf{T}f
\longrightarrow
u_G
\longrightarrow
Tu_G.
$$

This is structurally very close to the existing P1/P2 transformation path.